<a href="https://colab.research.google.com/github/Aisyaaswy/Pemrograman_Mesin/blob/main/jobsheet_6/HNSW/praktikum_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM 3**


In [1]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679801 sha256=4ef0928a4ca9c3703c94c44d1223f80275dba2540a31d6c56374544c22a672d1
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


Percobaan berikut akan membandingkan exact NN dengan HNSW pada 1000 data 2D.

In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")


=== Exact NN ===
Indices: [[423 867 307 659 277]]
Distances: [[0.02273295 0.02654514 0.02888647 0.02907326 0.03236225]]
Waktu: 0.06359457969665527 detik

=== HNSW ===
Indices: [[423 867 307 659 277]]
Distances: [[0.00051679 0.00070464 0.00083443 0.00084525 0.00104731]]
Waktu: 0.00026869773864746094 detik


Lakukan percobaan pada metric distance yang berbeda, 1000 vs 1jt data, 2D vs 5D data. catat hasilnya pada tabel yang anda buat sendiri seperti pada praktikum 1.

In [3]:
# Konfigurasi: (Jumlah Data, Dimensi)
configs = [
    (1000, 2),
    (1000000, 2),
    (1000, 5),
    (1000000, 5)
]

hasil_eksperimen = []
k = 5 # Cari 5 tetangga terdekat

for num_elements, dim in configs:
    # 1. Buat data acak (Seed dikunci agar hasilnya konsisten)
    np.random.seed(42)
    data = np.random.random((num_elements, dim)).astype(np.float32)
    query = np.random.random((1, dim)).astype(np.float32)

    # 2. Exact NN (Brute Force)
    # Catatan: Jarak Inner Product pada hnswlib dihitung dengan rumus (1.0 - dot_product)
    start_exact = time.time()
    exact_distances = 1.0 - np.dot(data, query[0])
    exact_indices = np.argsort(exact_distances)[:k]
    time_exact = time.time() - start_exact

    # 3. HNSW (Approximate NN)
    # Inisialisasi index HNSW dengan space 'ip' (Inner Product)
    p = hnswlib.Index(space='ip', dim=dim)
    p.init_index(max_elements=num_elements, ef_construction=100, M=16)
    p.add_items(data)
    p.set_ef(50)

    start_hnsw = time.time()
    labels, distances = p.knn_query(query, k=k)
    time_hnsw = time.time() - start_hnsw

    # 4. Format hasil index agar rapi (pakai spasi, tanpa koma di dalam kurung siku)
    str_idx_exact = "[" + " ".join(map(str, exact_indices)) + "]"
    str_idx_hnsw = "[" + " ".join(map(str, labels[0])) + "]"

    # Simpan hasil
    hasil_eksperimen.append({
        'Metric Distance': 'Inner Product (IP)',
        'Dimensi': f"{dim}D",
        'Jumlah Data': num_elements,
        'Hasil Index terdekat (ENN vs HNSW)': f"{str_idx_exact} , {str_idx_hnsw}",
        'Waktu komputasi (ENN vs HNSW) detik': f"{time_exact:.5f} , {time_hnsw:.5f}"
    })

# 5. Cetak Tabel Hasil
df_hasil = pd.DataFrame(hasil_eksperimen)
pd.set_option('display.max_colwidth', None)
display(df_hasil)

,Metric Distance,Dimensi,Jumlah Data,Hasil Index terdekat (ENN vs HNSW),Waktu komputasi (ENN vs HNSW) detik
0,Inner Product (IP),2D,1000,"[249 712 598 977 932] , [249 712 598 977 932]","0.01979 , 0.00012"
1,Inner Product (IP),2D,1000000,"[489250 265642 659640 28687 702931] , [489250 265642 659640 28687 702931]","0.36248 , 0.00017"
2,Inner Product (IP),5D,1000,"[210 106 49 220 486] , [210 106 49 220 486]","0.00017 , 0.00010"
3,Inner Product (IP),5D,1000000,"[958577 238618 509273 52984 409344] , [958577 238618 509273 52984 409344]","0.11889 , 0.00014"
